In [ ]:
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.26" trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",
    "unsloth/gemma-7b-it-bnb-4bit", # Instruct version of Gemma 7b
    "unsloth/gemma-2b-bnb-4bit",
    "unsloth/gemma-2b-it-bnb-4bit", # Instruct version of Gemma 2b
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2b-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

In [33]:
# how to read json file with pandas
import pandas as pd
df = pd.read_json('/content/CSE4078S24_Grp7_AlpacaStyle_Dataset1.json')
df

,instruction,input,output
0,Aşağıdaki ifadeye karşılık ver.,"Merhaba, nasılsın?",Ben iyiyim. Sen nasılsın?
1,Aşağıdaki ifadeye karşılık ver.,Ben iyiyim. Sen nasılsın?,Oldukça iyiyim. Sorduğun için teşekkürler.
2,Aşağıdaki ifadeye karşılık ver.,Oldukça iyiyim. Sorduğun için teşekkürler.,Sorun değil. Nasılsın?
3,Aşağıdaki ifadeye karşılık ver.,Sorun değil. Nasılsın?,Ben harikaydım. Ya sen?
4,Aşağıdaki ifadeye karşılık ver.,Ben harikaydım. Ya sen?,İyiydim. Şu anda okuldayım.
...,...,...,...
3720,Aşağıdaki ifadeye karşılık ver.,Bu iyi bir soru. Belki de yaşlılıktan değildir.,Sağ elini mi kullanıyorsun?
3721,Aşağıdaki ifadeye karşılık ver.,Sağ elini mi kullanıyorsun?,Evet. Tüm hayatım boyunca.
3722,Aşağıdaki ifadeye karşılık ver.,Evet. Tüm hayatım boyunca.,Sağ elini yıpratıyorsun. Bu kadar çok kullanma...
3723,Aşağıdaki ifadeye karşılık ver.,Sağ elini yıpratıyorsun. Bu kadar çok kullanma...,ama bütün yazılarımı sağ elimle yazarım.


In [34]:
alpaca = {
    'instruction': "",
    'input': "",
    'output':""
}

In [35]:
# open a new dataframe
df_new = pd.DataFrame()

In [36]:
def get_token_length(text):

  # Tokenize the text
  tokens = tokenizer.tokenize(text)

  # Get the token length
  token_length = len(tokens)

  return token_length

In [ ]:
# read df row by row and concat it to another df

for index, row in df.iterrows():
  if get_token_length(alpaca['instruction']) > 512:
    alpaca = {
    'instruction': "",
    'input': "",
    'output':""
    }
    print(f'At index {index}')

  if index == 0 or alpaca['instruction'] == "":
    alpaca['instruction'] = alpaca['instruction'] + 'Kullanıcı : '  + row['input'] + '\n'
    alpaca['output'] = row['output']
  else:
    # check if index is divisble by two
    if index % 2 == 1:
      alpaca['instruction'] = alpaca['instruction'] + 'Ajan : '  + row['input'] + '\n'
      alpaca['output'] = row['output']
    else:
      alpaca['instruction'] = alpaca['instruction'] + 'Kullanıcı : '  + row['input'] + '\n'
      alpaca['output'] = row['output']
  # concat this alpaca to a dataframe
  df_new = pd.concat([df_new, pd.DataFrame([alpaca])])


In [44]:
df_new

,instruction,input,output
0,"Kullanıcı : Merhaba, nasılsın?\n\n Yukarıdaki ...",,Ben iyiyim. Sen nasılsın?
1,"Kullanıcı : Merhaba, nasılsın?\nAjan : Ben iyi...",,Oldukça iyiyim. Sorduğun için teşekkürler.
2,"Kullanıcı : Merhaba, nasılsın?\nAjan : Ben iyi...",,Sorun değil. Nasılsın?
3,"Kullanıcı : Merhaba, nasılsın?\nAjan : Ben iyi...",,Ben harikaydım. Ya sen?
4,"Kullanıcı : Merhaba, nasılsın?\nAjan : Ben iyi...",,İyiydim. Şu anda okuldayım.
...,...,...,...
3720,Kullanıcı : Bu hiç iyi değil.\nKullanıcı : Baz...,,Sağ elini mi kullanıyorsun?
3721,Kullanıcı : Bu hiç iyi değil.\nKullanıcı : Baz...,,Evet. Tüm hayatım boyunca.
3722,Kullanıcı : Bu hiç iyi değil.\nKullanıcı : Baz...,,Sağ elini yıpratıyorsun. Bu kadar çok kullanma...
3723,Kullanıcı : Bu hiç iyi değil.\nKullanıcı : Baz...,,ama bütün yazılarımı sağ elimle yazarım.


In [39]:
# add a sentence to instruction column of each row of df_new
df_new['instruction'] = df_new['instruction'] + '\n Yukarıdaki konuşma geçmişine göre sohbeti devam ettir. '

In [ ]:
# reset dataframe index
df_new = df_new.reset_index(drop=True)
df_new

In [ ]:
# read df_new row by row
json_list = []
for index, row in df_new.iterrows():
  json_list.append(
{
    'instruction': row['instruction'],
    'input': row['input'],
    'output':row['output']
})
json_list

In [43]:
# save a list that contains json to file
import json
with open('json_list.json', 'w', encoding='utf-8') as file:
    json.dump(json_list, file, ensure_ascii=False)